# 02 — BDD100K Preparation for YOLO Training

**Goal:** Convert BDD100K detection annotations to YOLO format and build the dataset structure.

This notebook:
1. Reads BDD100K detection JSON annotations
2. Converts to YOLO format (`class_id x_center y_center width height`)
3. Creates `images/train`, `images/val`, `labels/train`, `labels/val` locally on Colab SSD
4. Generates the dataset YAML for training
5. Visualises sample annotations
6. Prints class mapping and distribution

## 1 — Install Dependencies

In [ ]:
!pip install -q ultralytics opencv-python matplotlib Pillow pyyaml tqdm

## 2 — Mount Drive & Configure Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sys

# ── Path configuration ─────────────────────────────────────────────
ECOCAR_ROOT = "/content/drive/MyDrive/EcoCAR"
BDD_RAW_DIR = os.path.join(ECOCAR_ROOT, "datasets", "bdd100k_raw")

# 💡 CRITICAL SPEED OPTIMIZATION:
# Creating symlinks or labels directly inside Google Drive is EXTREMELY SLOW.
# We create the training dataset locally inside Colab's fast NVMe storage.
YOLO_DATASET_DIR = "/content/bdd100k_yolo"

# ── Debug mode ─────────────────────────────────────────────────────
DEBUG_LIMIT = None  # Change to 100 for fast debugging

if DEBUG_LIMIT:
    print(f"⚡ DEBUG MODE: Processing only {DEBUG_LIMIT} images")

print(f"Raw BDD dir:     {BDD_RAW_DIR}")
print(f"YOLO dataset:    {YOLO_DATASET_DIR}  (Colab Local SSD)")

## 3 — Clone/Upload Source Utilities

In [ ]:
# Option B: Clone from git
!git clone https://github.com/Siyun-Chen/EcoCAR-Perception-Pipeline-YOLO26-BDD100K.git /content/ecocar
PROJECT_SRC = "/content/ecocar/src"

if os.path.isdir(PROJECT_SRC):
    sys.path.insert(0, os.path.dirname(PROJECT_SRC))
    print(f"✅ Added {PROJECT_SRC} to path")
else:
    print(f"⚠️ Source dir not found at {PROJECT_SRC}")
    print(f"   The required functions are defined inline below as fallback.")

In [ ]:
# Try to import from src, fall back to inline definitions
try:
    from src.dataset_utils import (
        BDD_CLASSES, CLASS_TO_ID, ID_TO_CLASS,
        convert_bdd100k_to_yolo, create_dataset_yaml,
        verify_dataset_structure, link_or_copy_images,
        print_class_distribution, get_bdd_class_mapping,
        find_expected_images,
    )
    print("✅ Imported from src.dataset_utils")
except ImportError:
    print("⚠️ Could not import src.dataset_utils, using inline definitions")

    import json
    import shutil
    from pathlib import Path
    import yaml
    from tqdm import tqdm

    BDD_CLASSES = [
        "person", "rider", "car", "truck", "bus",
        "train", "motorcycle", "bicycle", "traffic light", "traffic sign",
    ]
    CLASS_TO_ID = {name: idx for idx, name in enumerate(BDD_CLASSES)}
    ID_TO_CLASS = {idx: name for idx, name in enumerate(BDD_CLASSES)}

    def get_bdd_class_mapping():
        return BDD_CLASSES, CLASS_TO_ID, ID_TO_CLASS

    def convert_bdd100k_to_yolo(json_path, output_label_dir, img_width=1280, img_height=720, debug_limit=None):
        os.makedirs(output_label_dir, exist_ok=True)
        with open(json_path, 'r') as f:
            data = json.load(f)
        if debug_limit:
            data = data[:debug_limit]
        class_counts = {c: 0 for c in BDD_CLASSES}
        for frame in tqdm(data, desc="Converting"):
            img_name = frame['name']
            stem = Path(img_name).stem
            label_name = stem + '.txt'
            label_path = os.path.join(output_label_dir, label_name)
            lines = []
            for label in (frame.get('labels') or []):
                cat = label.get('category', '')
                if cat not in CLASS_TO_ID:
                    continue
                box = label.get('box2d')
                if not box:
                    continue
                x1, y1, x2, y2 = box['x1'], box['y1'], box['x2'], box['y2']
                xc = max(0, min(1, ((x1+x2)/2) / img_width))
                yc = max(0, min(1, ((y1+y2)/2) / img_height))
                w = max(0, min(1, (x2-x1) / img_width))
                h = max(0, min(1, (y2-y1) / img_height))
                lines.append(f"{CLASS_TO_ID[cat]} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
                class_counts[cat] += 1
            with open(label_path, 'w') as f:
                f.write('\n'.join(lines))
        return class_counts

    def create_dataset_yaml(dataset_root, output_path, train_images='images/train', val_images='images/val'):
        config = {'path': dataset_root, 'train': train_images, 'val': val_images,
                  'nc': len(BDD_CLASSES), 'names': {i: n for i, n in enumerate(BDD_CLASSES)}}
        os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
        with open(output_path, 'w') as f:
            yaml.dump(config, f, default_flow_style=False, sort_keys=False)
        print(f"✅ YAML written to: {output_path}")
        return os.path.abspath(output_path)

    def verify_dataset_structure(dataset_root):
        all_ok = True
        for sub in ['images/train','images/val','labels/train','labels/val']:
            full = os.path.join(dataset_root, sub)
            if not os.path.isdir(full):
                print(f"❌ Missing: {full}"); all_ok = False
            else:
                c = len(os.listdir(full))
                print(f"✅ {sub}: {c} files")
                if c == 0: all_ok = False
        return all_ok

    def link_or_copy_images(src_dir, dst_dir, use_symlinks=True, debug_limit=None, image_list=None):
        os.makedirs(dst_dir, exist_ok=True)
        if image_list is not None:
            files = image_list
        else:
            try:
                files = sorted([f for f in os.listdir(src_dir)
                                if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            except OSError:
                print(f"⚠ OSError listing {src_dir}.")
                return 0
        if debug_limit:
            files = files[:debug_limit]
        count = 0
        missing = 0
        for f in tqdm(files, desc=f"→ {os.path.basename(dst_dir)}"):
            src = os.path.join(src_dir, f)
            dst = os.path.join(dst_dir, f)
            if not os.path.exists(src):
                missing += 1; continue
            if os.path.exists(dst):
                count += 1; continue
            ok = False
            try:
                if use_symlinks:
                    os.symlink(src, dst)
                else:
                    shutil.copy2(src, dst)
                ok = True
            except OSError:
                try:
                    shutil.copy2(src, dst)
                    ok = True
                except FileNotFoundError:
                    ok = False
            if ok and os.path.exists(dst):
                count += 1
            else:
                missing += 1
        print(f"Linked/copied: {count}, Missing/skipped: {missing}")
        return count

    def find_expected_images(label_dir, image_src_dirs):
        """Find images matching label files, searching multiple candidate directories.

        Uses os.listdir() ONCE per directory to build an in-memory set,
        then matches labels against it. MUCH faster than per-file os.path.exists()
        on Google Drive FUSE. Symlink dirs are deprioritized (FUSE errors).

        Returns: (list_of_image_filenames, source_directory)
        """
        if isinstance(image_src_dirs, str):
            image_src_dirs = [image_src_dirs]

        # Deprioritize symlinks (they cause FUSE errors on Google Drive)
        real_dirs = [d for d in image_src_dirs if os.path.isdir(d) and not os.path.islink(d)]
        link_dirs = [d for d in image_src_dirs if os.path.isdir(d) and os.path.islink(d)]
        ordered_dirs = real_dirs + link_dirs

        if not ordered_dirs:
            print(f"⚠ No valid image source directories found among: {image_src_dirs}")
            return [], ""

        # Index each directory's image files ONCE (fast even on FUSE)
        dir_image_sets = {}
        for d in ordered_dirs:
            try:
                files = os.listdir(d)
                stem_map = {}
                for f in files:
                    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                        stem = os.path.splitext(f)[0]
                        if stem not in stem_map:
                            stem_map[stem] = f
                dir_image_sets[d] = stem_map
                if stem_map:
                    print(f"  📁 Indexed {len(stem_map)} images from: {d}")
            except OSError as e:
                print(f"  ⚠ Cannot list {d}: {e}")
                continue

        if not dir_image_sets:
            print("⚠ Could not list any image directories")
            return [], ""

        # Get label stems and match in memory
        label_files = sorted([f for f in os.listdir(label_dir) if f.endswith('.txt')])
        label_stems = [os.path.splitext(f)[0] for f in label_files]

        expected = []
        chosen_dir = ""
        for stem in label_stems:
            for d in ordered_dirs:
                if d not in dir_image_sets:
                    continue
                if stem in dir_image_sets[d]:
                    expected.append(dir_image_sets[d][stem])
                    if not chosen_dir:
                        chosen_dir = d
                    break

        return expected, chosen_dir

    def print_class_distribution(class_counts):
        total = sum(class_counts.values())
        print(f"\n{'Class':<20} {'Count':>8} {'Pct':>7}")
        print('─'*37)
        for name in BDD_CLASSES:
            c = class_counts.get(name, 0)
            pct = (c/total*100) if total else 0
            print(f"{name:<20} {c:>8,} {pct:>6.1f}%")
        print('─'*37)
        print(f"{'TOTAL':<20} {total:>8,}")

## 4 — Locate BDD100K Files

Attempts to find label JSONs and image directories automatically.

**Priority order:**
1. `paths_config.yaml` saved by notebook 01 (most reliable)
2. Standard BDD100K directory layout (`bdd100k/labels/`, `bdd100k/images/100k/`)
3. Flat extract layout (`100k/train/`, `labels/`)

In [ ]:
import glob
import yaml

# ── Try loading paths from notebook 01's config first ─────────────
paths_cfg_path = os.path.join(ECOCAR_ROOT, 'paths_config.yaml')
TRAIN_LABEL = None
VAL_LABEL = None
BDD_IMAGES_BASE = None

if os.path.isfile(paths_cfg_path):
    with open(paths_cfg_path, 'r') as f:
        pcfg = yaml.safe_load(f)
    for key in pcfg:
        val = pcfg[key]
        if isinstance(val, str) and os.path.isfile(val):
            if 'train' in key and val.endswith('.json'):
                TRAIN_LABEL = val
            elif 'val' in key and val.endswith('.json'):
                VAL_LABEL = val
    # Check for image dir — VERIFY it actually exists and is NOT a stale path
    for key in ['train_images_dir']:
        if key in pcfg:
            candidate_dir = pcfg[key]
            if os.path.isdir(candidate_dir) and not os.path.islink(candidate_dir):
                BDD_IMAGES_BASE = os.path.dirname(candidate_dir)
            else:
                print(f"⚠️ Config path '{candidate_dir}' is stale or a symlink, using fallback")
    if TRAIN_LABEL:
        print(f'✅ Loaded label paths from {paths_cfg_path}')

# ── Fallback: auto-detect label paths ─────────────────────────────
if not TRAIN_LABEL or not VAL_LABEL:
    label_search_paths = [
        os.path.join(BDD_RAW_DIR, 'bdd100k', 'labels', 'det_20', 'det_train.json'),
        os.path.join(BDD_RAW_DIR, 'bdd100k', 'labels', 'det_20', 'det_val.json'),
        os.path.join(BDD_RAW_DIR, 'bdd100k', 'labels', 'bdd100k_labels_images_train.json'),
        os.path.join(BDD_RAW_DIR, 'bdd100k', 'labels', 'bdd100k_labels_images_val.json'),
        os.path.join(BDD_RAW_DIR, 'labels', 'det_20', 'det_train.json'),
        os.path.join(BDD_RAW_DIR, 'labels', 'det_20', 'det_val.json'),
        os.path.join(BDD_RAW_DIR, 'labels', 'bdd100k_labels_images_train.json'),
        os.path.join(BDD_RAW_DIR, 'labels', 'bdd100k_labels_images_val.json'),
    ]
    for p in label_search_paths:
        if os.path.isfile(p):
            if 'train' in os.path.basename(p) and not TRAIN_LABEL:
                TRAIN_LABEL = p
            elif 'val' in os.path.basename(p) and not VAL_LABEL:
                VAL_LABEL = p

# ── Fallback: auto-detect image paths ─────────────────────────────
# Prefer REAL directories over symlinks (Drive FUSE is unreliable with symlinks)
if not BDD_IMAGES_BASE:
    img_search = [
        os.path.join(BDD_RAW_DIR, '100k'),                       # flat extract (REAL dir, preferred)
        os.path.join(BDD_RAW_DIR, 'images', '100k'),             # alt layout
        os.path.join(BDD_RAW_DIR, 'bdd100k', 'images', '100k'),  # standard BDD layout (may be symlinks)
        '/content/bdd100k_images/100k',                           # local symlinks from nb01
    ]
    for p in img_search:
        if os.path.isdir(p) and (
            os.path.isdir(os.path.join(p, 'train')) or
            os.path.isdir(os.path.join(p, 'val'))
        ):
            BDD_IMAGES_BASE = p
            break

print(f"Train labels: {TRAIN_LABEL}")
print(f"Val labels:   {VAL_LABEL}")
print(f"Images base:  {BDD_IMAGES_BASE}")

if not all([TRAIN_LABEL, VAL_LABEL, BDD_IMAGES_BASE]):
    print('\n❌ Could not auto-detect all paths.')
    print('   Set TRAIN_LABEL, VAL_LABEL, BDD_IMAGES_BASE manually below.')
else:
    print('\n✅ All paths detected!')

## 5 — Convert Annotations to YOLO Format

In [ ]:
# Create YOLO dataset directories locally
for split in ['train', 'val']:
    os.makedirs(os.path.join(YOLO_DATASET_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(YOLO_DATASET_DIR, 'labels', split), exist_ok=True)

print("📁 YOLO local dataset directories created")

In [ ]:
# Convert train labels
print("\n" + "="*50)
print(" Converting TRAIN labels (FAST SSD I/O)")
print("="*50)

train_counts = convert_bdd100k_to_yolo(
    json_path=TRAIN_LABEL,
    output_label_dir=os.path.join(YOLO_DATASET_DIR, 'labels', 'train'),
    debug_limit=DEBUG_LIMIT,
)

print("\nTrain class distribution:")
print_class_distribution(train_counts)

In [ ]:
# Convert val labels
print("\n" + "="*50)
print(" Converting VAL labels")
print("="*50)

val_counts = convert_bdd100k_to_yolo(
    json_path=VAL_LABEL,
    output_label_dir=os.path.join(YOLO_DATASET_DIR, 'labels', 'val'),
    debug_limit=DEBUG_LIMIT,
)

print("\nVal class distribution:")
print_class_distribution(val_counts)

## 6 — Link/Copy Images

### ⚠️ FUSE-Safe Design
- Uses `os.listdir()` ONCE per directory to build an in-memory lookup set
- Matches labels against this set (pure Python, no FUSE calls per file)
- Symlink directories are **deprioritized** — real dirs come first
- Val images are also searched in train source (since nb01's 85/15 split keeps all images in one folder)

In [ ]:
# ── Build candidate image source directories ──────────────────────
# Lists ALL possible places where raw BDD100K images might live.
# Real (non-symlink) directories come FIRST — FUSE is unreliable with symlinks.
print("=" * 60)
print(" IMAGE SOURCE CANDIDATES")
print("=" * 60)

CANDIDATE_IMG_DIRS = []

# All possible image source locations
all_candidates = []
if BDD_IMAGES_BASE:
    all_candidates.extend([
        os.path.join(BDD_IMAGES_BASE, 'train'),
        os.path.join(BDD_IMAGES_BASE, 'val'),
    ])
extra = [
    os.path.join(BDD_RAW_DIR, '100k', 'train'),
    os.path.join(BDD_RAW_DIR, '100k', 'val'),
    os.path.join(BDD_RAW_DIR, 'bdd100k', 'images', '100k', 'train'),
    os.path.join(BDD_RAW_DIR, 'bdd100k', 'images', '100k', 'val'),
    os.path.join(BDD_RAW_DIR, 'images', '100k', 'train'),
    os.path.join(BDD_RAW_DIR, 'images', '100k', 'val'),
]
for c in extra:
    if c not in all_candidates:
        all_candidates.append(c)

# Separate real dirs from symlinks, then combine (real first)
for d in all_candidates:
    if os.path.isdir(d) and not os.path.islink(d):
        CANDIDATE_IMG_DIRS.append(d)
        try:
            img_count = sum(1 for f in os.listdir(d) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
            print(f"  ✅ REAL DIR: {d}  ({img_count} images)")
        except OSError:
            print(f"  ⚠️ REAL DIR: {d}  (exists but listdir failed)")

for d in all_candidates:
    if os.path.isdir(d) and os.path.islink(d) and d not in CANDIDATE_IMG_DIRS:
        CANDIDATE_IMG_DIRS.append(d)
        target = os.readlink(d)
        print(f"  🔗 SYMLINK: {d} → {target}  (deprioritized)")

for d in all_candidates:
    if not os.path.isdir(d) and d not in CANDIDATE_IMG_DIRS:
        print(f"  ❌ MISSING: {d}")

In [ ]:
# ── Link TRAIN images ──────────────────────────────────────────────
train_label_dir = os.path.join(YOLO_DATASET_DIR, 'labels', 'train')
train_img_dst = os.path.join(YOLO_DATASET_DIR, 'images', 'train')

print(f"\nSearching for train images in {len(CANDIDATE_IMG_DIRS)} candidate dirs...")
expected_train_imgs, train_src_dir = find_expected_images(
    train_label_dir,
    CANDIDATE_IMG_DIRS,
)

print(f"\n✅ Found {len(expected_train_imgs)} matching train images")
if train_src_dir:
    print(f"   Source: {train_src_dir}")

n_train = 0
if expected_train_imgs and train_src_dir:
    n_train = link_or_copy_images(
        train_src_dir, train_img_dst,
        use_symlinks=True,
        debug_limit=DEBUG_LIMIT,
        image_list=expected_train_imgs,
    )
print(f"✅ {n_train} train images linked")

In [ ]:
# ── Link VAL images ────────────────────────────────────────────────
# IMPORTANT: val images may live in the SAME folder as train images
# (notebook 01 created an 85/15 split from one folder), so we search
# all candidate dirs including train sources.
val_label_dir = os.path.join(YOLO_DATASET_DIR, 'labels', 'val')
val_img_dst = os.path.join(YOLO_DATASET_DIR, 'images', 'val')

print(f"\nSearching for val images in {len(CANDIDATE_IMG_DIRS)} candidate dirs...")
expected_val_imgs, val_src_dir = find_expected_images(
    val_label_dir,
    CANDIDATE_IMG_DIRS,
)

print(f"\n✅ Found {len(expected_val_imgs)} matching val images")
if val_src_dir:
    print(f"   Source: {val_src_dir}")

n_val = 0
if expected_val_imgs and val_src_dir:
    n_val = link_or_copy_images(
        val_src_dir, val_img_dst,
        use_symlinks=True,
        debug_limit=DEBUG_LIMIT,
        image_list=expected_val_imgs,
    )
print(f"✅ {n_val} val images linked")

## 7 — Generate Dataset YAML

In [ ]:
yaml_path = os.path.join(YOLO_DATASET_DIR, 'bdd100k_yolo.yaml')
yaml_backup = os.path.join(ECOCAR_ROOT, "datasets", "bdd100k_yolo.yaml")

yaml_file = create_dataset_yaml(
    dataset_root=YOLO_DATASET_DIR,
    output_path=yaml_path,
)

import shutil
shutil.copy2(yaml_path, yaml_backup)
print(f"✅ YAML backed up to Drive at: {yaml_backup}")

print("\nGenerated YAML:")
with open(yaml_path, 'r') as f:
    print(f.read())

## 8 — Verify Dataset Structure

In [ ]:
print("\n" + "="*50)
print(" DATASET VERIFICATION")
print("="*50)

ok = verify_dataset_structure(YOLO_DATASET_DIR)

if ok:
    print("\n🎯 Dataset ready for training!")
else:
    print("\n⚠️ Fix the issues above before training.")

## 9 — Visualise Sample Annotations

In [ ]:
import cv2
import matplotlib.pyplot as plt
import random

train_labels_dir = os.path.join(YOLO_DATASET_DIR, 'labels', 'train')
train_images_dir = os.path.join(YOLO_DATASET_DIR, 'images', 'train')

label_files = [f for f in os.listdir(train_labels_dir) if f.endswith('.txt')]
sample_labels = random.sample(label_files, min(4, len(label_files)))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

colors = [
    (255,80,80), (255,160,60), (60,180,255), (100,100,255), (255,220,60),
    (180,100,255), (100,255,100), (255,100,200), (0,255,200), (200,200,0),
]

for idx, lf in enumerate(sample_labels):
    img_name = lf.replace('.txt', '.jpg')
    img_path = os.path.join(train_images_dir, img_name)
    label_path = os.path.join(train_labels_dir, lf)

    img = cv2.imread(img_path)
    if img is None:
        img_path = img_path.replace('.jpg', '.png')
        img = cv2.imread(img_path)
    if img is None:
        axes[idx].set_title(f"Could not load {img_name}")
        continue

    h, w = img.shape[:2]
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5: continue
            cls_id = int(parts[0])
            xc, yc, bw, bh = [float(v) for v in parts[1:5]]
            x1 = int((xc - bw/2) * w)
            y1 = int((yc - bh/2) * h)
            x2 = int((xc + bw/2) * w)
            y2 = int((yc + bh/2) * h)
            color = colors[cls_id % len(colors)]
            cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
            cv2.putText(img, BDD_CLASSES[cls_id], (x1, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    axes[idx].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[idx].set_title(img_name, fontsize=9)
    axes[idx].axis('off')

plt.suptitle('Sample BDD100K Annotations (YOLO format)', fontsize=13)
plt.tight_layout()
plt.show()

## 10 — Print Class Mapping

In [ ]:
print("\nBDD100K Detection Class Mapping:")
print("="*35)
for idx, name in enumerate(BDD_CLASSES):
    print(f"  {idx:>2} → {name}")
print("="*35)
print(f"Total classes: {len(BDD_CLASSES)}")

## 11 — (Optional) Lane Annotation Preview

In [ ]:
import json

lane_candidates = glob.glob(os.path.join(BDD_RAW_DIR, '**', '*lane*'), recursive=True)

if lane_candidates:
    print("🔍 Lane annotation files found:")
    for lf in lane_candidates[:5]:
        print(f"   {lf}")
    lane_json = [f for f in lane_candidates if f.endswith('.json')]
    if lane_json:
        with open(lane_json[0], 'r') as f:
            lane_data = json.load(f)
        if isinstance(lane_data, list) and len(lane_data) > 0:
            print(f"\nLane file entries: {len(lane_data)}")
            print(f"Sample entry keys: {list(lane_data[0].keys())}")
            print(f"\nSample (truncated):")
            print(json.dumps(lane_data[0], indent=2)[:500])
        print("\n📌 Lane annotations will be used in future lane-marking notebooks.")
else:
    print("ℹ️ No standalone lane annotation files found.")
    print("   (Lane poly2d annotations may be embedded in the merged label JSONs — this is normal.")
    print("    Notebook 07 will extract them for lane mask preparation.)")

## 12 — Backup Dataset to Drive

In [ ]:
import shutil

DRIVE_DATASET_DIR = os.path.join(ECOCAR_ROOT, "datasets", "bdd100k_yolo")

# Backup images to Drive
for split in ['train', 'val']:
    src = os.path.join(YOLO_DATASET_DIR, 'images', split)
    dst = os.path.join(DRIVE_DATASET_DIR, 'images', split)
    if os.path.isdir(src) and len(os.listdir(src)) > 0:
        os.makedirs(dst, exist_ok=True)
        count = 0
        for f in os.listdir(src):
            src_file = os.path.join(src, f)
            dst_file = os.path.join(dst, f)
            if not os.path.exists(dst_file):
                if os.path.islink(src_file):
                    real_src = os.path.realpath(src_file)
                    if os.path.exists(real_src):
                        shutil.copy2(real_src, dst_file)
                        count += 1
                else:
                    shutil.copy2(src_file, dst_file)
                    count += 1
        print(f"✅ Backed up {count} new {split} images to Drive")
    else:
        print(f"⚠️ No {split} images to back up")

# Backup labels
for split in ['train', 'val']:
    src = os.path.join(YOLO_DATASET_DIR, 'labels', split)
    dst = os.path.join(DRIVE_DATASET_DIR, 'labels', split)
    if os.path.isdir(src):
        os.makedirs(dst, exist_ok=True)
        !cp -n {src}/*.txt {dst}/ 2>/dev/null || true
        print(f"✅ Backed up {split} labels to Drive")

print("\n✅ Dataset backed up to Drive at:", DRIVE_DATASET_DIR)

## 13 — Summary

In [ ]:
print("\n" + "="*60)
print(" BDD100K PREPARATION — COMPLETE")
print("="*60)
print(f" YOLO dataset dir:  {YOLO_DATASET_DIR} (Colab Fast SSD)")
print(f" Dataset YAML:      {yaml_path}")
print(f" Train images:      {n_train}")
print(f" Val images:        {n_val}")
print(f" Classes:           {len(BDD_CLASSES)}")
if DEBUG_LIMIT:
    print(f" ⚡ Debug mode:     {DEBUG_LIMIT} images only")
print("="*60)
print("\n🎯 Proceed to notebook 03 for fine-tuning YOLO26 on BDD100K!")